# 🎬 TH-Labs AI Dubbing — Colab (free T4 GPU) runner

Runs the **full** pipeline — Whisper · NLLB-200 · edge-tts · **OpenVoice** voice-cloning · Demucs — on Colab's free **T4 GPU**, and serves the Studio UI on a **public URL** via a Cloudflare quick tunnel (no account, no ngrok token).

The React app is built to static files and served by the FastAPI backend, so the whole thing is **one port behind one tunnel**.

> ⚠️ **Ephemeral.** Colab ends the session on idle (~90 min) or after ~12 h, and you get a **new URL** each run. This is a you-driven GPU demo — for an always-on link use Hugging Face Spaces instead.

---
### Before you run
1. **Runtime → Change runtime type → T4 GPU**, then **Save**.
2. Push your latest code to GitHub (from your machine):
   ```
   git add -A && git commit -m "Studio-only + serve frontend from FastAPI" && git push
   ```
3. Run every cell top-to-bottom. The **last cell prints your public URL**.

In [ ]:
# 1 · Confirm a GPU is attached.  Errors here → Runtime → Change runtime type → T4 GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2 · Clone the repo.  It's PRIVATE, so paste a GitHub token (scope: repo) when
#     prompted.  If you've made the repo public, just press Enter to skip the token.
import getpass, os, subprocess

REPO   = "asilbekali/TH-Labs-full"   # owner/name
BRANCH = "main"
DEST   = "/content/TH-Labs-full"

token = getpass.getpass(f"GitHub token for {REPO} (Enter if public): ").strip()
url   = f"https://{token + '@' if token else ''}github.com/{REPO}.git"
subprocess.run(["rm", "-rf", DEST], check=False)
rc = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, DEST]).returncode
del token, url          # don't keep the token around
assert rc == 0, "clone failed — check the token / repo name / branch"
os.chdir(DEST)
print("✔ cloned into", DEST)

In [ ]:
# 3 · System packages: ffmpeg + Node (usually already on Colab — this just ensures it).
!ffmpeg -version >/dev/null 2>&1 && echo "ffmpeg ✔" || (apt-get -qq update && apt-get -qq install -y ffmpeg)
!node --version >/dev/null 2>&1 || (curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/dev/null 2>&1 && apt-get -qq install -y nodejs)
!echo "node $(node --version) · npm $(npm --version)"

In [ ]:
# 4 · Python ML stack.  Colab already ships CUDA-enabled torch; these keep it.
#     First run ~2–4 min.
%pip install -q -r backend/requirements.txt
%pip install -q openai-whisper "transformers>=4.40" sentencepiece edge-tts soundfile silero-vad demucs huggingface_hub
# OpenVoice v2 tone-color converter (voice cloning).  --no-deps skips its old pins;
# we add just the text/audio helpers its runtime actually imports.
%pip install -q "git+https://github.com/myshell-ai/OpenVoice.git" --no-deps
%pip install -q unidecode inflect eng_to_ipa pypinyin cn2an jieba wavmark librosa
import torch
print(f"✔ deps ready · torch {torch.__version__} · CUDA available: {torch.cuda.is_available()}")

In [ ]:
# 5 · OpenVoice v2 converter checkpoint (131 MB) → the path the backend expects.
import os, shutil
from huggingface_hub import hf_hub_download

dst = "backend/models/openvoice_v2/converter"
os.makedirs(dst, exist_ok=True)
for name in ("config.json", "checkpoint.pth"):
    src = hf_hub_download("myshell-ai/OpenVoiceV2", f"converter/{name}")
    shutil.copy(src, os.path.join(dst, name))
print("✔ OpenVoice converter:", os.listdir(dst))

In [ ]:
# 6 · Build the React UI → frontend/dist (FastAPI serves it).  ~1–2 min.
import os, subprocess
subprocess.run("npm ci --no-audit --no-fund || npm install --no-audit --no-fund",
               cwd="frontend", shell=True, check=True)
subprocess.run("npm run build", cwd="frontend", shell=True, check=True)
print("✔ frontend built →", os.path.exists("frontend/dist/index.html"))

In [ ]:
# 7 · Cloudflare quick-tunnel binary (no account needed).
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared && cloudflared --version

In [ ]:
# 8 · Launch the GPU backend + tunnel.  KEEP THIS CELL RUNNING (and the tab open).
import os, re, sys, time, subprocess, threading, urllib.request

os.environ.update({
    "TH_LABS_MODE":              "auto",     # real models where installed
    "TH_LABS_WHISPER_MODEL":     "medium",   # paper-grade; a T4 handles it
    "TH_LABS_WHISPER_DEVICE":    "cuda",
    "TH_LABS_CLONE_DEVICE":      "cuda",      # OpenVoice cloning on GPU
    "TH_LABS_SEPARATION_DEVICE": "cuda",      # Demucs on GPU
})

# FastAPI (no --reload → one clean process)
api = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd="backend", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
threading.Thread(target=lambda: [print("[api]", ln, end="") for ln in api.stdout], daemon=True).start()

for _ in range(60):                          # wait for the port to bind
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=2); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("backend did not start — scroll up for [api] errors")
print("✔ backend up on :8000")

# Cloudflare tunnel → grab the public URL
tun = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
public = None
for ln in tun.stdout:
    m = re.search(r"https://[-\w.]+\.trycloudflare\.com", ln)
    if m:
        public = m.group(0); break
threading.Thread(target=lambda: [None for _ in tun.stdout], daemon=True).start()  # drain

print("\n" + "=" * 70)
print("  🎬  OPEN YOUR DUBBING STUDIO:")
print("      " + str(public))
print("=" * 70)
print("\nFirst load downloads the models (~5–6 GB, a few minutes). The navbar")
print('badge flips to "N/4 AI live" when they are warm. Leave this cell running.')

## Using it
- Open the URL above. Try the **built-in sample** (no upload) for a quick EN→UZ run, or upload a short clip.
- **Quality:** *Fast* (base model, skips separation) is quickest; *Studio* (Whisper-medium + Demucs + OpenVoice cloning) is best — the GPU makes it usable here.
- **Voice cloning** (OpenVoice) runs on the GPU and reports the *measured* speaker similarity. **OmniVoice** (16 GB) does **not** fit a free T4 — you'd need an A100 (Colab Pro).

## Keeping it alive / stopping
- Keep the tab open; Colab reclaims idle GPUs.
- Stop with **Runtime → Interrupt**, or run `!pkill -f cloudflared; pkill -f uvicorn`.
- Re-running the launch cell issues a **fresh URL**.